In [1]:
import requests

url = "https://api.nbp.pl/api/exchangerates/tables/C"
response = requests.get(url)
response.raise_for_status()
response_json = response.json()

response_json


[{'table': 'C',
  'no': '145/C/NBP/2026',
  'tradingDate': '2026-07-28',
  'effectiveDate': '2026-07-29',
  'rates': [{'currency': 'dolar amerykański',
    'code': 'USD',
    'bid': 3.7655,
    'ask': 3.8415},
   {'currency': 'dolar australijski',
    'code': 'AUD',
    'bid': 2.6251,
    'ask': 2.6781},
   {'currency': 'dolar kanadyjski',
    'code': 'CAD',
    'bid': 2.6707,
    'ask': 2.7247},
   {'currency': 'euro', 'code': 'EUR', 'bid': 4.2822, 'ask': 4.3688},
   {'currency': 'forint (Węgry)',
    'code': 'HUF',
    'bid': 0.011856,
    'ask': 0.012096},
   {'currency': 'frank szwajcarski',
    'code': 'CHF',
    'bid': 4.5991,
    'ask': 4.6921},
   {'currency': 'funt szterling', 'code': 'GBP', 'bid': 5.0085, 'ask': 5.1097},
   {'currency': 'jen (Japonia)',
    'code': 'JPY',
    'bid': 0.022983,
    'ask': 0.023447},
   {'currency': 'korona czeska', 'code': 'CZK', 'bid': 0.1771, 'ask': 0.1807},
   {'currency': 'korona duńska', 'code': 'DKK', 'bid': 0.5728, 'ask': 0.5844},
   {'c

In [25]:
# Transform and save to file

import json
from datetime import date

currencies = response_json[0]["rates"]
date_of_read = response_json[0]["effectiveDate"]

source_path = f"../data/{date_of_read}.json"

for currency in currencies:
    currency["ingest_date"] = date_of_read
    currency["path_to_source"] = source_path

with open(f"../data/{date.today()}.json", 'w') as f:
    f.write(json.dumps(currencies, indent=2))


In [ ]:
# Model data
from sqlalchemy import Column, Integer, String, Float, Date
from sqlalchemy.orm import declarative_base

Base = declarative_base()

class Currency(Base):
    __tablename__ = "Currency"

    id = Column(Integer, primary_key=True)
    currency = Column(String)
    code = Column(String)
    bid = Column(Float)
    ask = Column(Float)
    ingest_date = Column(Date)
    path_to_source= Column(String)

In [ ]:
from sqlalchemy import create_engine
from config import config
db = config("database.ini")

DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{db['user']}:{db['password']}"
    f"@{db['host']}/"
    f"{db['database']}"
)
engine = create_engine(DATABASE_URL)


In [ ]:
from sqlalchemy.orm import sessionmaker
from sqlalchemy import insert, inspect

engine = create_engine(DATABASE_URL)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

inspector = inspect(engine)
inspector.get_table_names()

session = SessionLocal()


In [ ]:
# Add all data to table 
new_rates = [Currency(currency=exchange_rate["currency"],
                         code=exchange_rate["code"],
                         bid=exchange_rate["bid"],
                         ask=exchange_rate["ask"],
                         ingest_date=exchange_rate["ingest_date"],
                         path_to_source=exchange_rate["path_to_source"]) for exchange_rate in currencies]
    
session.add_all(new_rates)
session.commit()

In [ ]:
# Removing all data from table 

session.query(Currency).delete()
session.commit()